[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Indexes &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell, which seeds two hundred thousand documents so the
numbers mean something. Run it first. Each task starts from `only_id`, so they can be run in any
order.


In [1]:
import datetime as dt
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

def failed(error):
    """The part of a failure that is the same on every run. An index build failure names two
    fresh uuids before the real message, and the real message is after 'caused by'."""
    details = getattr(error, "details", None) or {}
    message = details.get("errmsg", str(error).split(", full error")[0])
    return f"{type(error).__name__}: {message.split(' :: caused by :: ')[-1]}"


def plan(cursor):
    """How the server answered: the stage it used and how much it had to look at."""
    explained = cursor.explain()
    winner = explained["queryPlanner"]["winningPlan"]
    stats = explained["executionStats"]
    return {"stage": winner.get("inputStage", winner).get("stage"),
            "index keys": stats["totalKeysExamined"],
            "documents": stats["totalDocsExamined"],
            "returned": stats["nReturned"]}


def only_id(collection):
    """Drop every index but the one on _id, so a section can start from nothing."""
    for index in list(collection.list_indexes()):
        if index["name"] != "_id_":
            collection.drop_index(index["name"])
    return [index["name"] for index in collection.list_indexes()]


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(200_000), "products   <- this notebook needs a big collection")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  200000 products   <- this notebook needs a big collection
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 200000


**1.** With nothing to help it.


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
print("indexes:", only_id(shop.products))

print("maker = Aster:", plan(shop.products.find({"maker": "Aster"})))
client.close()


indexes: ['_id_']
maker = Aster: {'stage': 'COLLSCAN', 'index keys': 0, 'documents': 200000, 'returned': 50000}


Two hundred thousand documents read to return fifty thousand. The ratio is four to one, which is
what a collection scan looks like when the answer is a quarter of the collection.


**2.** With an index.


In [3]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
only_id(shop.products)

shop.products.create_index("maker", name="maker_1")
print("maker = Aster:", plan(shop.products.find({"maker": "Aster"})))
print("every key read turned into a row of the answer")
client.close()


maker = Aster: {'stage': 'IXSCAN', 'index keys': 50000, 'documents': 50000, 'returned': 50000}
every key read turned into a row of the answer


`index keys`, `documents` and `returned` are now the same number. The server touched nothing it did
not need.


**3.** The prefix rule.


In [4]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
only_id(shop.products)

shop.products.create_index([("maker", 1), ("stock", 1)], name="maker_stock")
print("maker and stock:", plan(shop.products.find({"maker": "Aster",
                                                   "stock": {"$lt": 10}})))
print("stock alone:    ", plan(shop.products.find({"stock": {"$lt": 10}})))
client.close()


maker and stock: {'stage': 'IXSCAN', 'index keys': 1195, 'documents': 1195, 'returned': 1195}
stock alone:     {'stage': 'COLLSCAN', 'index keys': 0, 'documents': 200000, 'returned': 4956}


The index is sorted by `maker` first, so the low stock values are scattered through it in four
separate runs and there is no way to walk to them. A query on the second field alone gets nothing.


**4.** Two regexes, one index.


In [5]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
only_id(shop.products)
shop.products.create_index("name", name="name_1")

print("anchored:  ", plan(shop.products.find({"name": {"$regex": "^Aster laptop 12"}})))
print("unanchored:", plan(shop.products.find({"name": {"$regex": "laptop 12"}})))
client.close()


anchored:   {'stage': 'IXSCAN', 'index keys': 557, 'documents': 556, 'returned': 556}
unanchored: {'stage': 'IXSCAN', 'index keys': 200000, 'documents': 2222, 'returned': 2222}


Both say `IXSCAN`. The unanchored one read every key in the index, which is the whole collection, so
the stage name told you nothing and the numbers told you everything.


**5.** A constraint that is also an index.


In [6]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
only_id(shop.products)

shop.products.create_index("sku", name="sku_1", unique=True)
existing = shop.products.find_one({"_id": 0})["sku"]

try:
    shop.products.insert_one({"_id": 10 ** 9, "sku": existing})
except pymongo.errors.DuplicateKeyError as error:
    print(failed(error))

print("still", shop.products.count_documents({}), "documents")
only_id(shop.products)
client.close()


DuplicateKeyError: E11000 duplicate key error collection: shop.products index: sku_1 dup key: { sku: "LAP-000000" }
still 200000 documents


The seeded skus are already distinct, so the index builds. Had two products shared one, the build
itself would have failed and the constraint would never have existed.


**6.** Expired, and still there.


In [7]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

shop.answers.drop()
shop.answers.create_index("at", expireAfterSeconds=1)
shop.answers.insert_one({"at": dt.datetime.now(dt.timezone.utc) - dt.timedelta(days=7)})

print("a week past its expiry:", shop.answers.count_documents({}))
time.sleep(3)
print("three seconds later:   ", shop.answers.count_documents({}))
print("the sweep runs about once a minute, so this is expected")
shop.answers.drop()
client.close()


a week past its expiry: 1
three seconds later:    1
the sweep runs about once a minute, so this is expected


A TTL index is a promise that the document will go away, not that it has. Any code that must not see
an expired document has to say so in its filter.


---

&#8592; **Back to:** [Indexes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/08-indexes.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
